In [1]:
import polars as pl
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import date, timedelta
from collections import defaultdict
from itertools import product as iterproduct

DATA_DIR = Path('/workspace/data/raw')
CANDIDATE_DIR = Path('/workspace/outputs/candidates')
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

TD_A, TD_B, TD_C, TD_D = 2.5e4, 1.5e5, 2e-1, 1e3
EVAL_CUTOFFS = [12, 24, 48, 96, 120]

# ============================================================
# データ読み込み・整数変換
# ============================================================
transactions_raw = pl.read_csv(
    DATA_DIR / 'transactions_train.csv',
    schema_overrides={'article_id': pl.String, 'customer_id': pl.String}
).with_columns(pl.col('t_dat').str.to_date())

customers_raw = pl.read_csv(
    DATA_DIR / 'customers.csv',
    schema_overrides={'customer_id': pl.String}
)

articles_raw = pl.read_csv(
    DATA_DIR / 'articles.csv',
    schema_overrides={'article_id': pl.String}
)

# 整数変換マップ
print("整数変換マップ作成中...")
all_customer_ids = pl.concat([
    transactions_raw.select('customer_id'),
    customers_raw.select('customer_id')
]).unique().sort('customer_id')['customer_id'].to_list()

all_article_ids = pl.concat([
    transactions_raw.select('article_id'),
    articles_raw.select('article_id')
]).unique().sort('article_id')['article_id'].to_list()

customer_id_map = {v: i for i, v in enumerate(all_customer_ids)}
customer_id_reverse = {i: v for v, i in customer_id_map.items()}
article_id_map = {v: i for i, v in enumerate(all_article_ids)}
article_id_reverse = {i: v for v, i in article_id_map.items()}

# 整数変換
transactions = (
    transactions_raw
    .with_columns([
        pl.col('customer_id').replace_strict(customer_id_map, default=None).cast(pl.Int32),
        pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32),
    ])
    .drop_nulls()
)

customers = (
    customers_raw
    .with_columns([
        pl.col('customer_id').replace_strict(customer_id_map, default=None).cast(pl.Int32),
        pl.when(pl.col('age').is_null()).then(-1)
        .otherwise((pl.col('age') / 10).cast(pl.Int32) * 10)
        .alias('age_bin'),
        pl.col('age').fill_null(-1).cast(pl.Int32),
    ])
    .drop_nulls(subset=['customer_id'])
)

articles = (
    articles_raw
    .with_columns(
        pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32)
    )
    .drop_nulls(subset=['article_id'])
)

CATEGORY_COLS = ['section_name', 'garment_group_name', 'index_group_name']
articles_category = articles_raw.with_columns(
    pl.col('article_id').replace_strict(article_id_map, default=None).cast(pl.Int32)
).drop_nulls(subset=['article_id']).select(['article_id'] + CATEGORY_COLS)

last_ts = transactions['t_dat'].max()
print(f"全データ最終日: {last_ts}")
print(f"customer数: {len(customer_id_map):,}")
print(f"article数: {len(article_id_map):,}")
print("整数変換完了!")

整数変換マップ作成中...
全データ最終日: 2020-09-22
customer数: 1,371,980
article数: 105,542
整数変換完了!


In [2]:
# ============================================================
# 共通関数
# ============================================================
def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

def save_candidates(df, name, cutoff):
    path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
    df.write_parquet(path)
    print(f"  保存: {path}")
    return df

def load_candidates(name, cutoff):
    path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
    if path.exists():
        print(f"  読み込み: {path}")
        return pl.read_parquet(path)
    return None

def normalize_scores(cand_dict):
    """各ユーザーのスコアを0〜1に正規化"""
    normalized = {}
    for cid, cands in cand_dict.items():
        if not cands:
            continue
        max_score = max(s for _, s, _ in cands)
        min_score = min(s for _, s, _ in cands)
        diff = max_score - min_score
        if diff == 0:
            normalized[cid] = [(art, 1.0, rank) for art, s, rank in cands]
        else:
            normalized[cid] = [(art, (s - min_score) / diff, rank) for art, s, rank in cands]
    return normalized

def evaluate_candidates(cand_dicts, weights, actuals, valid_customers, k_list=EVAL_CUTOFFS):
    """重み付きスコアでRecall@KとMAP@Kを計算"""
    metric_acc = {k: {'recall': [], 'map': []} for k in k_list}
    
    for cid in valid_customers:
        actual_set = actuals[cid]
        
        score_dict = {}
        for name, w in weights.items():
            if w == 0 or name not in cand_dicts:
                continue
            for art, score, rank in cand_dicts[name].get(cid, []):
                score_dict[art] = score_dict.get(art, 0) + w * score
        
        sorted_cands = sorted(score_dict.items(), key=lambda x: -x[1])
        prediction = [art for art, _ in sorted_cands]
        
        for k in k_list:
            hits = len(set(prediction[:k]) & actual_set)
            metric_acc[k]['recall'].append(hits / len(actual_set) if actual_set else 0.0)
            metric_acc[k]['map'].append(apk(actual_set, prediction, k))
    
    results = {}
    for k in k_list:
        results[k] = {
            'recall': np.mean(metric_acc[k]['recall']),
            'map': np.mean(metric_acc[k]['map']),
        }
    return results

print("共通関数定義完了")

共通関数定義完了


In [3]:
# ============================================================
# 候補生成関数
# ============================================================

def build_timedecay(transactions, cutoff, top_n=120):
    name = 'timedecay'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    TD_A / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    TD_B * (-TD_C * pl.col('days_ago').cast(pl.Float64)).exp() - TD_D
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    result = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_repurchase(transactions, cutoff, top_n=120):
    name = 'repurchase'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    result = (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg(pl.max('t_dat').alias('last_date'))
        .with_columns(
            pl.col('last_date').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .with_columns(
            (1.0 / (1.0 + (pl.lit(cutoff) - pl.col('last_date')).dt.total_days().cast(pl.Float64))).alias('score')
        )
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_variant(transactions, cutoff, timedecay_df=None, top_n=30):
    name = 'variant'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    if timedecay_df is None:
        timedecay_df = load_candidates('timedecay', cutoff)
    articles_product = articles.select(['article_id', 'product_code'])
    result = (
        timedecay_df
        .filter(pl.col('rank') <= 50)
        .select(['customer_id', 'article_id', 'score'])
        .join(articles_product, on='article_id', how='left')
        .join(
            articles_product.rename({'article_id': 'variant_article_id'}),
            on='product_code', how='left'
        )
        .filter(pl.col('article_id') != pl.col('variant_article_id'))
        .drop_nulls(subset=['variant_article_id'])
        .group_by(['customer_id', pl.col('variant_article_id').alias('article_id')])
        .agg(pl.col('score').sum().alias('score'))
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_also_bought(transactions, cutoff, top_n=20, history_days=180):
    name = 'also_bought'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中（直近{history_days}日・月ごと分割）...")
    start_date = cutoff - timedelta(days=history_days)
    df_train = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) & (pl.col('t_dat') <= pl.lit(cutoff))
    )
    months = (
        df_train.select(pl.col('t_dat').dt.truncate('1mo').alias('month'))
        .unique().sort('month').to_series().to_list()
    )
    print(f"    対象月数: {len(months)}")
    all_pairs = []
    for i, month in enumerate(months):
        month_end = month + timedelta(days=31)
        df_month = df_train.filter(
            (pl.col('t_dat') >= pl.lit(month)) & (pl.col('t_dat') < pl.lit(month_end))
        )
        if len(df_month) == 0:
            continue
        pairs = (
            df_month.select(['customer_id', 't_dat', 'article_id'])
            .join(df_month.select(['customer_id', 't_dat', 'article_id']), on=['customer_id', 't_dat'], how='inner')
            .filter(pl.col('article_id') != pl.col('article_id_right'))
            .group_by(['article_id', 'article_id_right'])
            .agg(pl.len().alias('count'))
        )
        all_pairs.append(pairs)
        print(f"    進捗: {i+1}/{len(months)}ヶ月")
    combined = (
        pl.concat(all_pairs)
        .group_by(['article_id', 'article_id_right'])
        .agg(pl.col('count').sum())
        .filter(pl.col('count') >= 10)
        .sort(['article_id', 'count'], descending=[False, True])
    )
    also_bought_dict = defaultdict(list)
    for row in combined.iter_rows(named=True):
        also_bought_dict[row['article_id']].append((row['article_id_right'], row['count']))
    recent_start = cutoff - timedelta(days=30)
    user_history = (
        transactions.filter(
            (pl.col('t_dat') >= pl.lit(recent_start)) & (pl.col('t_dat') <= pl.lit(cutoff))
        )
        .group_by('customer_id')
        .agg(pl.col('article_id').alias('articles'))
    )
    rows_list = []
    for row in user_history.iter_rows(named=True):
        cid = row['customer_id']
        hist_arts = row['articles']
        purchased_set = set(hist_arts)
        candidates = {}
        for art in hist_arts:
            for paired_art, count in also_bought_dict.get(art, []):
                if paired_art not in purchased_set:
                    candidates[paired_art] = candidates.get(paired_art, 0) + count
        sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
        for rank, (art, score) in enumerate(sorted_cands, start=1):
            rows_list.append({'customer_id': cid, 'article_id': art, 'score': float(score), 'rank': rank})
    result = pl.DataFrame(rows_list).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.col('rank').cast(pl.Int32),
    ])
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_item2item_cf(transactions, cutoff, top_n=20):
    name = 'item2item_cf'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    from scipy.sparse import csr_matrix
    from sklearn.preprocessing import normalize
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    user_item = df_train.group_by(['customer_id', 'article_id']).agg(pl.len().alias('count'))
    n_users = len(customer_id_map)
    n_items = len(article_id_map)
    row = user_item['customer_id'].to_numpy()
    col = user_item['article_id'].to_numpy()
    data = user_item['count'].to_numpy().astype(np.float32)
    uim = csr_matrix((data, (row, col)), shape=(n_users, n_items))
    iim_norm = normalize(uim.T.tocsr(), norm='l2')
    print(f"    行列サイズ: {iim_norm.shape}")
    start_date = cutoff - timedelta(days=30)
    recent_articles = (
        df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
        .select('article_id').unique().to_series().to_list()
    )
    print(f"    直近1ヶ月の購買商品数: {len(recent_articles):,}")
    print("    類似度辞書を事前計算中...")
    similar_items_dict = {}
    for i, art in enumerate(recent_articles):
        if i % 10000 == 0:
            print(f"      進捗: {i:,} / {len(recent_articles):,}")
        if art >= iim_norm.shape[0]:
            continue
        sim_scores = iim_norm[art].dot(iim_norm.T).toarray().flatten()
        top_items = np.argsort(sim_scores)[::-1][1:top_n+1]
        similar_items_dict[art] = [
            (int(idx), float(sim_scores[idx]))
            for idx in top_items if sim_scores[idx] > 0
        ]
    print(f"    類似度辞書完成: {len(similar_items_dict):,}商品")
    recent_hist = (
        df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('articles'))
    )
    rows_list = []
    for i, row_data in enumerate(recent_hist.iter_rows(named=True)):
        if i % 100000 == 0:
            print(f"      進捗: {i:,} / {len(recent_hist):,}")
        cid = row_data['customer_id']
        hist_arts = row_data['articles']
        purchased_set = set(hist_arts)
        candidates = {}
        for art in hist_arts:
            for sim_art, sim_score in similar_items_dict.get(art, []):
                if sim_art not in purchased_set:
                    candidates[sim_art] = candidates.get(sim_art, 0) + sim_score
        sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
        for rank, (art, score) in enumerate(sorted_cands, start=1):
            rows_list.append({'customer_id': cid, 'article_id': art, 'score': score, 'rank': rank})
    result = pl.DataFrame(rows_list).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.col('rank').cast(pl.Int32),
    ])
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_category_popular(transactions, cutoff, top_n=20):
    name = 'category_popular'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    start_date = cutoff - timedelta(days=30)
    recent = df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
    category_top = {}
    for col in CATEGORY_COLS:
        recent_with_cat = recent.join(
            articles_category.select(['article_id', col]), on='article_id', how='left'
        )
        counts = (
            recent_with_cat.filter(pl.col(col).is_not_null())
            .group_by([col, 'article_id']).agg(pl.len().alias('count'))
            .sort([col, 'count'], descending=[False, True])
        )
        top_for_col = defaultdict(list)
        for row in counts.iter_rows(named=True):
            cat = row[col]
            if len(top_for_col[cat]) < top_n:
                top_for_col[cat].append((row['article_id'], row['count']))
        category_top[col] = dict(top_for_col)
    history_with_cat = (
        df_train.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=90)))
        .join(articles_category, on='article_id', how='left')
    )
    rows_list = []
    for col in CATEGORY_COLS:
        user_cat_counts = (
            history_with_cat.filter(pl.col(col).is_not_null())
            .group_by(['customer_id', col]).agg(pl.len().alias('count'))
            .sort(['customer_id', 'count'], descending=[False, True])
        )
        user_top_cats = defaultdict(list)
        for row in user_cat_counts.iter_rows(named=True):
            cid = row['customer_id']
            if len(user_top_cats[cid]) < 2:
                user_top_cats[cid].append(row[col])
        purchased = (
            df_train.group_by(['customer_id', 'article_id']).agg(pl.len())
            .group_by('customer_id').agg(pl.col('article_id').alias('articles'))
        )
        purchased_dict = {
            row['customer_id']: set(row['articles'])
            for row in purchased.iter_rows(named=True)
        }
        for cid, cats in user_top_cats.items():
            purchased_set = purchased_dict.get(cid, set())
            candidates = {}
            for cat in cats:
                for art, count in category_top[col].get(cat, []):
                    if art not in purchased_set:
                        candidates[art] = candidates.get(art, 0) + count
            sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
            for rank, (art, score) in enumerate(sorted_cands, start=1):
                rows_list.append({'customer_id': cid, 'article_id': art, 'score': float(score), 'rank': rank})
    result = (
        pl.DataFrame(rows_list)
        .group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum(), pl.col('rank').min())
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_bin_popular(transactions, cutoff, top_n=50):
    name = 'bin_popular'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    recent = (
        df_train
        .filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
        .join(customers.select(['customer_id', 'age_bin']), on='customer_id', how='left')
    )
    bin_popular = (
        recent.group_by(['age_bin', 'article_id']).agg(pl.len().alias('count'))
        .with_columns(
            pl.col('count').rank(method='ordinal', descending=True).over('age_bin').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
    )
    result = (
        customers.select(['customer_id', 'age_bin'])
        .join(bin_popular.select(['age_bin', 'article_id', 'count', 'rank']), on='age_bin', how='left')
        .drop_nulls()
        .with_columns(pl.col('count').cast(pl.Float64).alias('score'))
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_general_pred(transactions, cutoff, top_n=50):
    name = 'general_pred'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train_td = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train_td.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train_td = df_train_td.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train_td = (
        df_train_td
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
    )
    general_arts = (
        df_train_td.group_by('article_id')
        .agg(pl.col('quotient').sum().alias('score'))
        .sort('score', descending=True)
        .head(top_n)
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).cast(pl.Int32).alias('rank')
        )
        .select(['article_id', 'score', 'rank'])
        .with_columns([
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    all_customers_df = customers.select('customer_id').with_columns(pl.lit(1).alias('key'))
    general_arts_key = general_arts.with_columns(pl.lit(1).alias('key'))
    result = (
        all_customers_df.join(general_arts_key, on='key', how='inner')
        .drop('key')
        .with_columns(pl.col('customer_id').cast(pl.Int32))
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_purchase_interval(transactions, cutoff, top_n=20):
    name = 'purchase_interval'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    purchase_dates = (
        df_train
        .sort(['customer_id', 'article_id', 't_dat'])
        .group_by(['customer_id', 'article_id'])
        .agg([
            pl.col('t_dat').sort().alias('dates'),
            pl.col('t_dat').max().alias('last_date'),
            pl.len().alias('purchase_count')
        ])
        .filter(pl.col('purchase_count') >= 2)
    )
    rows_list = []
    for row in purchase_dates.iter_rows(named=True):
        dates = row['dates']
        if len(dates) < 2:
            continue
        diffs = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
        avg_interval = sum(diffs) / len(diffs)
        days_since_last = (cutoff - row['last_date']).days
        score = 1.0 / (1.0 + abs(days_since_last - avg_interval))
        rows_list.append({
            'customer_id': row['customer_id'],
            'article_id': row['article_id'],
            'score': score,
        })
    result = (
        pl.DataFrame(rows_list)
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_user_category_popular(transactions, cutoff, top_n=20):
    name = 'user_category_popular'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    start_date = cutoff - timedelta(days=30)
    recent = df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
    recent_with_cat = recent.join(
        articles.select(['article_id', 'product_type_no', 'garment_group_no']),
        on='article_id', how='left'
    )
    product_type_popular = (
        recent_with_cat.filter(pl.col('product_type_no').is_not_null())
        .group_by(['product_type_no', 'article_id']).agg(pl.len().alias('count'))
        .with_columns(
            pl.col('count').rank(method='ordinal', descending=True).over('product_type_no').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
    )
    hist_with_cat = (
        df_train.filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=90)))
        .join(articles.select(['article_id', 'product_type_no']), on='article_id', how='left')
        .filter(pl.col('product_type_no').is_not_null())
    )
    user_top_product_types = (
        hist_with_cat.group_by(['customer_id', 'product_type_no']).agg(pl.len().alias('count'))
        .with_columns(
            pl.col('count').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= 3)
        .select(['customer_id', 'product_type_no'])
    )
    result = (
        user_top_product_types
        .join(
            product_type_popular.select(['product_type_no', 'article_id', 'count', 'rank']),
            on='product_type_no', how='left'
        )
        .drop_nulls()
        .group_by(['customer_id', 'article_id'])
        .agg(pl.col('count').sum().alias('score'), pl.col('rank').min().alias('rank'))
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

print("候補生成関数定義完了")

候補生成関数定義完了


In [ ]:
# ============================================================
# グリッドサーチ（段階的探索）
# ============================================================
print("\n" + "="*60)
print("【グリッドサーチ（段階的探索）】")
print("="*60)

logic_names = list(candidate_dfs.keys())

# ============================================================
# STEP1: 粗い探索（全ロジック対象・粗い重み）
# ============================================================
print("\n[STEP1] 粗い探索...")
coarse_weights = [0.0, 1.0, 2.0, 3.0]

# 全ロジックを対象に単体で重みを変えて探索
step1_results = {}
for name in logic_names:
    best_w = 0.0
    best_r = 0.0
    for w in coarse_weights:
        weights = {n: 1.0 for n in logic_names}
        weights[name] = w
        results = evaluate_candidates(cand_dicts, weights, actuals, valid_customers)
        r = results[12]['recall']
        if r > best_r:
            best_r = r
            best_w = w
    step1_results[name] = {'best_w': best_w, 'best_recall': best_r}
    print(f"  {name:<25} best_w: {best_w:.2f} Recall@12: {best_r:.6f}")

# STEP1の最良重みを設定
step1_weights = {name: res['best_w'] for name, res in step1_results.items()}
print(f"\nSTEP1最良重み: {step1_weights}")
step1_eval = evaluate_candidates(cand_dicts, step1_weights, actuals, valid_customers)
print(f"STEP1 Recall@12: {step1_eval[12]['recall']:.6f}")




【グリッドサーチ（段階的探索）】

[STEP1] 粗い探索...
  timedecay                 best_w: 3.00 Recall@12: 0.056483
  repurchase                best_w: 2.00 Recall@12: 0.055147
  variant                   best_w: 1.00 Recall@12: 0.053965
  also_bought               best_w: 1.00 Recall@12: 0.053965
  item2item_cf              best_w: 1.00 Recall@12: 0.053965
  category_popular          best_w: 0.00 Recall@12: 0.055263
  bin_popular               best_w: 1.00 Recall@12: 0.053965
  general_pred              best_w: 0.00 Recall@12: 0.054647
  purchase_interval         best_w: 1.00 Recall@12: 0.053965
  user_category_popular     best_w: 0.00 Recall@12: 0.054656

STEP1最良重み: {'timedecay': 3.0, 'repurchase': 2.0, 'variant': 1.0, 'also_bought': 1.0, 'item2item_cf': 1.0, 'category_popular': 0.0, 'bin_popular': 1.0, 'general_pred': 0.0, 'purchase_interval': 1.0, 'user_category_popular': 0.0}
STEP1 Recall@12: 0.056388

[STEP2] 細かい探索...
細かく探索するロジック: ['timedecay', 'category_popular', 'repurchase', 'user_category_popula

KeyboardInterrupt: 

In [6]:
# ============================================================
# STEP2: 細かい探索（top4ロジックをbest±0.5で探索）
# ============================================================
print("\n[STEP2] 細かい探索（best±0.5）...")

# STEP1の最良重みから上位4ロジックを特定
top4_names = ['timedecay', 'repurchase', 'variant', 'also_bought']

# 各ロジックの探索範囲（best±0.5、ただし0以上）
step2_weight_ranges = {}
for name in top4_names:
    best_w = step1_weights[name]
    w_minus = max(0.0, best_w - 0.5)
    w_plus = best_w + 0.5
    step2_weight_ranges[name] = [w_minus, best_w, w_plus]
    print(f"  {name:<25} 探索範囲: {step2_weight_ranges[name]}")

total = 3 ** len(top4_names)
print(f"\n組み合わせ数: {total:,}")

best_recall = step1_eval[12]['recall']
best_weights = step1_weights.copy()
best_results = step1_eval

for i, combo in enumerate(iterproduct(*[step2_weight_ranges[n] for n in top4_names])):
    if all(w == 0 for w in combo):
        continue
    
    # その他のロジックはSTEP1の最良重みを使用
    weights = {name: step1_weights[name] for name in logic_names if name not in top4_names}
    weights.update(dict(zip(top4_names, combo)))
    
    results = evaluate_candidates(cand_dicts, weights, actuals, valid_customers)
    recall_12 = results[12]['recall']
    
    if recall_12 > best_recall:
        best_recall = recall_12
        best_weights = weights.copy()
        best_results = results
        print(f"  [{i+1}/{total}] 新ベスト Recall@12: {best_recall:.6f} | {dict(zip(top4_names, combo))}")

print(f"\nSTEP2完了!")
print(f"最良Recall@12: {best_recall:.6f}")
print(f"最良重み: {best_weights}")

# ============================================================
# 最良重みでの全K評価
# ============================================================
print("\n" + "="*60)
print("【最良重みでの評価】")
print("="*60)
print(f"{'k':>6} {'recall':>10} {'MAP':>10}")
print("-"*30)
for k in EVAL_CUTOFFS:
    print(f"{k:>6} {best_results[k]['recall']:>10.6f} {best_results[k]['map']:>10.6f}")

# ============================================================
# best_weightsを保存
# ============================================================
weights_path = CANDIDATE_DIR / f"best_weights_{CUTOFF}.json"
with open(weights_path, 'w') as f:
    json.dump({'weights': best_weights, 'recall_12': best_recall}, f, indent=2)
print(f"\nbest_weights保存: {weights_path}")


[STEP2] 細かい探索（best±0.5）...
  timedecay                 探索範囲: [2.5, 3.0, 3.5]
  repurchase                探索範囲: [1.5, 2.0, 2.5]
  variant                   探索範囲: [0.5, 1.0, 1.5]
  also_bought               探索範囲: [0.5, 1.0, 1.5]

組み合わせ数: 81
  [1/81] 新ベスト Recall@12: 0.056749 | {'timedecay': 2.5, 'repurchase': 1.5, 'variant': 0.5, 'also_bought': 0.5}
  [2/81] 新ベスト Recall@12: 0.056792 | {'timedecay': 2.5, 'repurchase': 1.5, 'variant': 0.5, 'also_bought': 1.0}
  [3/81] 新ベスト Recall@12: 0.056842 | {'timedecay': 2.5, 'repurchase': 1.5, 'variant': 0.5, 'also_bought': 1.5}
  [5/81] 新ベスト Recall@12: 0.056905 | {'timedecay': 2.5, 'repurchase': 1.5, 'variant': 1.0, 'also_bought': 1.0}
  [29/81] 新ベスト Recall@12: 0.056978 | {'timedecay': 3.0, 'repurchase': 1.5, 'variant': 0.5, 'also_bought': 1.0}
  [30/81] 新ベスト Recall@12: 0.056989 | {'timedecay': 3.0, 'repurchase': 1.5, 'variant': 0.5, 'also_bought': 1.5}
  [32/81] 新ベスト Recall@12: 0.057106 | {'timedecay': 3.0, 'repurchase': 1.5, 'variant': 1.0, 'also_b

In [7]:
# ============================================================
# STEP3: item2item_cf, bin_popular, purchase_intervalを±0.5で探索
# ============================================================
print("\n[STEP3] 細かい探索（item2item_cf, bin_popular, purchase_interval ±0.5）...")

step3_names = ['item2item_cf', 'bin_popular', 'purchase_interval']

step3_weight_ranges = {}
for name in step3_names:
    best_w = best_weights[name]
    w_minus = max(0.0, best_w - 0.5)
    w_plus = best_w + 0.5
    step3_weight_ranges[name] = [w_minus, best_w, w_plus]
    print(f"  {name:<25} 探索範囲: {step3_weight_ranges[name]}")

total = 3 ** len(step3_names)
print(f"\n組み合わせ数: {total:,}")

best_recall_s3 = best_recall
best_weights_s3 = best_weights.copy()
best_results_s3 = best_results

for i, combo in enumerate(iterproduct(*[step3_weight_ranges[n] for n in step3_names])):
    weights = best_weights.copy()
    weights.update(dict(zip(step3_names, combo)))
    
    results = evaluate_candidates(cand_dicts, weights, actuals, valid_customers)
    recall_12 = results[12]['recall']
    
    if recall_12 > best_recall_s3:
        best_recall_s3 = recall_12
        best_weights_s3 = weights.copy()
        best_results_s3 = results
        print(f"  [{i+1}/{total}] 新ベスト Recall@12: {best_recall_s3:.6f} | {dict(zip(step3_names, combo))}")

print(f"\nSTEP3完了!")
print(f"最良Recall@12: {best_recall_s3:.6f}")
print(f"最良重み: {best_weights_s3}")

# 全K評価
print("\n" + "="*60)
print("【最良重みでの評価（STEP3後）】")
print("="*60)
print(f"{'k':>6} {'recall':>10} {'MAP':>10}")
print("-"*30)
for k in EVAL_CUTOFFS:
    print(f"{k:>6} {best_results_s3[k]['recall']:>10.6f} {best_results_s3[k]['map']:>10.6f}")

# 保存
best_recall = best_recall_s3
best_weights = best_weights_s3
best_results = best_results_s3

weights_path = CANDIDATE_DIR / f"best_weights_{CUTOFF}.json"
with open(weights_path, 'w') as f:
    json.dump({'weights': best_weights, 'recall_12': best_recall}, f, indent=2)
print(f"\nbest_weights保存: {weights_path}")


[STEP3] 細かい探索（item2item_cf, bin_popular, purchase_interval ±0.5）...
  item2item_cf              探索範囲: [0.5, 1.0, 1.5]
  bin_popular               探索範囲: [0.5, 1.0, 1.5]
  purchase_interval         探索範囲: [0.5, 1.0, 1.5]

組み合わせ数: 27
  [4/27] 新ベスト Recall@12: 0.057456 | {'item2item_cf': 0.5, 'bin_popular': 1.0, 'purchase_interval': 0.5}
  [7/27] 新ベスト Recall@12: 0.057582 | {'item2item_cf': 0.5, 'bin_popular': 1.5, 'purchase_interval': 0.5}
  [16/27] 新ベスト Recall@12: 0.057602 | {'item2item_cf': 1.0, 'bin_popular': 1.5, 'purchase_interval': 0.5}
  [17/27] 新ベスト Recall@12: 0.057604 | {'item2item_cf': 1.0, 'bin_popular': 1.5, 'purchase_interval': 1.0}

STEP3完了!
最良Recall@12: 0.057604
最良重み: {'item2item_cf': 1.0, 'category_popular': 0.0, 'bin_popular': 1.5, 'general_pred': 0.0, 'purchase_interval': 1.0, 'user_category_popular': 0.0, 'timedecay': 3.5, 'repurchase': 1.5, 'variant': 1.0, 'also_bought': 1.0}

【最良重みでの評価（STEP3後）】
     k     recall        MAP
------------------------------
    12   0.05760

In [8]:
# ============================================================
# STEP1: 候補ファイル生成（cutoff=09-22）
# item2item_cfは最後に実行
# ============================================================
SUB_CUTOFF = transactions['t_dat'].max()
print(f"提出用cutoff: {SUB_CUTOFF}")

# item2item_cf以外を先に生成
logics_first = ['timedecay', 'repurchase', 'variant', 
                'also_bought', 'bin_popular', 'purchase_interval']

print("\n候補ファイル生成中（item2item_cf以外）...")
sub_cand_dfs = {}
for name in logics_first:
    print(f"\n  [{name}] 処理中...")
    cached = load_candidates(name, SUB_CUTOFF)
    if cached is None:
        if name == 'timedecay':
            cached = build_timedecay(transactions, SUB_CUTOFF)
        elif name == 'repurchase':
            cached = build_repurchase(transactions, SUB_CUTOFF)
        elif name == 'variant':
            td = load_candidates('timedecay', SUB_CUTOFF)
            cached = build_variant(transactions, SUB_CUTOFF, td)
        elif name == 'also_bought':
            cached = build_also_bought(transactions, SUB_CUTOFF)
        elif name == 'bin_popular':
            cached = build_bin_popular(transactions, SUB_CUTOFF)
        elif name == 'purchase_interval':
            cached = build_purchase_interval(transactions, SUB_CUTOFF)
    sub_cand_dfs[name] = cached
    print(f"  [{name}] 完了: {len(cached):,}行")

print("\nitem2item_cf以外の候補生成完了!")

# item2item_cfを最後に実行
print("\n  [item2item_cf] 処理中（時間がかかります）...")
cf_cached = load_candidates('item2item_cf', SUB_CUTOFF)
if cf_cached is None:
    cf_cached = build_item2item_cf(transactions, SUB_CUTOFF)
sub_cand_dfs['item2item_cf'] = cf_cached
print(f"  [item2item_cf] 完了: {len(cf_cached):,}行")

print("\n全候補ファイル生成完了!")
for name, df in sub_cand_dfs.items():
    print(f"  {name:<25} {len(df):,}行")

提出用cutoff: 2020-09-22

候補ファイル生成中（item2item_cf以外）...

  [timedecay] 処理中...
  読み込み: /workspace/outputs/candidates/timedecay_2020-09-22.parquet
  [timedecay] 完了: 10,077,536行

  [repurchase] 処理中...
  読み込み: /workspace/outputs/candidates/repurchase_2020-09-22.parquet
  [repurchase] 完了: 25,821,120行

  [variant] 処理中...
  読み込み: /workspace/outputs/candidates/variant_2020-09-22.parquet
  [variant] 完了: 23,170,472行

  [also_bought] 処理中...
  読み込み: /workspace/outputs/candidates/also_bought_2020-09-22.parquet
  [also_bought] 完了: 4,307,834行

  [bin_popular] 処理中...
  読み込み: /workspace/outputs/candidates/bin_popular_2020-09-22.parquet
  [bin_popular] 完了: 68,598,192行

  [purchase_interval] 処理中...
  読み込み: /workspace/outputs/candidates/purchase_interval_2020-09-22.parquet
  [purchase_interval] 完了: 3,014,713行

item2item_cf以外の候補生成完了!

  [item2item_cf] 処理中（時間がかかります）...
  読み込み: /workspace/outputs/candidates/item2item_cf_2020-09-22.parquet
  [item2item_cf] 完了: 5,012,145行

全候補ファイル生成完了!
  timedecay                 

In [10]:
# ============================================================
# STEP2: 重み付きスコア統合結果を保存
# ============================================================
active_logics = {
    'timedecay':         3.5,
    'repurchase':        1.5,
    'variant':           1.0,
    'also_bought':       1.0,
    'item2item_cf':      1.0,
    'bin_popular':       1.5,
    'purchase_interval': 1.0,
}

# print("候補dictに変換中...")
# sub_cand_dicts_raw = {}
# for name in active_logics.keys():
#     df = sub_cand_dfs[name]
#     d = defaultdict(list)
#     for row in df.iter_rows(named=True):
#         d[row['customer_id']].append((row['article_id'], row['score'], row['rank']))
#     sub_cand_dicts_raw[name] = dict(d)
#     print(f"  {name}: {len(d):,}ユーザー")

# print("スコア正規化中...")
# sub_cand_dicts = {name: normalize_scores(d) for name, d in sub_cand_dicts_raw.items()}

# # 全ユーザーの重み付きスコアを計算して保存
# print("\n重み付きスコア計算中...")
# all_customer_ids_list = list(customer_id_map.values())
# BATCH_SIZE = 100000
# all_rows = []

# for batch_start in tqdm(range(0, len(all_customer_ids_list), BATCH_SIZE), desc="スコア計算"):
#     batch = all_customer_ids_list[batch_start:batch_start+BATCH_SIZE]
    
#     for cid in batch:
#         score_dict = {}
#         for name, w in active_logics.items():
#             for art, score, rank in sub_cand_dicts[name].get(cid, []):
#                 score_dict[art] = score_dict.get(art, 0) + w * score
        
#         for art, score in score_dict.items():
#             all_rows.append({
#                 'customer_id': cid,
#                 'article_id': art,
#                 'candidate_score': score,
#             })

# # DataFrameに変換して保存
# print("DataFrameに変換中...")
# scored_df = pl.DataFrame(all_rows).with_columns([
#     pl.col('customer_id').cast(pl.Int32),
#     pl.col('article_id').cast(pl.Int32),
#     pl.col('candidate_score').cast(pl.Float32),
# ])

# print(f"全候補数: {len(scored_df):,}行")

# # parquetで保存
# scored_path = CANDIDATE_DIR / f"candidates_scored_{SUB_CUTOFF}.parquet"
# scored_df.write_parquet(scored_path)
# print(f"保存完了: {scored_path}")
# print(f"ユーザー数: {scored_df['customer_id'].n_unique():,}")
# print(f"ファイルサイズ: {scored_path.stat().st_size / 1024 / 1024:.1f} MB")

In [11]:
print("重み付きスコア計算中（Polars版）...")

# 各ロジックのスコアを正規化してからjoin
scored_list = []
for name, w in active_logics.items():
    df = sub_cand_dfs[name]
    
    # ユーザーごとにmin-max正規化
    normalized = (
        df.select(['customer_id', 'article_id', 'score'])
        .join(
            df.group_by('customer_id').agg([
                pl.col('score').max().alias('max_score'),
                pl.col('score').min().alias('min_score'),
            ]),
            on='customer_id', how='left'
        )
        .with_columns(
            pl.when(pl.col('max_score') == pl.col('min_score'))
            .then(1.0)
            .otherwise(
                (pl.col('score') - pl.col('min_score')) / 
                (pl.col('max_score') - pl.col('min_score'))
            )
            .alias('norm_score')
        )
        .with_columns(
            (pl.col('norm_score') * w).alias('weighted_score')
        )
        .select(['customer_id', 'article_id', 'weighted_score'])
    )
    scored_list.append(normalized)
    print(f"  {name}: 正規化完了")

# 全ロジックのスコアを結合して合算
print("スコア合算中...")
combined = pl.concat(scored_list)
scored_df = (
    combined
    .group_by(['customer_id', 'article_id'])
    .agg(pl.col('weighted_score').sum().alias('candidate_score'))
    .with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.col('candidate_score').cast(pl.Float32),
    ])
)

print(f"全候補数: {len(scored_df):,}行")

# parquetで保存
scored_path = CANDIDATE_DIR / f"candidates_scored_{SUB_CUTOFF}.parquet"
scored_df.write_parquet(scored_path)
print(f"保存完了: {scored_path}")
print(f"ユーザー数: {scored_df['customer_id'].n_unique():,}")
print(f"ファイルサイズ: {scored_path.stat().st_size / 1024 / 1024:.1f} MB")

重み付きスコア計算中（Polars版）...
  timedecay: 正規化完了
  repurchase: 正規化完了
  variant: 正規化完了
  also_bought: 正規化完了
  item2item_cf: 正規化完了
  bin_popular: 正規化完了
  purchase_interval: 正規化完了
スコア合算中...
全候補数: 122,600,347行
保存完了: /workspace/outputs/candidates/candidates_scored_2020-09-22.parquet
ユーザー数: 1,371,980
ファイルサイズ: 911.9 MB


In [12]:
print("提出ファイル作成中...")

# 上位12件を選択
submission = (
    scored_df
    .with_columns(
        pl.col('candidate_score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
    )
    .filter(pl.col('rank') <= 12)
    .sort(['customer_id', 'rank'])
    .group_by('customer_id')
    .agg(pl.col('article_id').alias('articles'))
    .with_columns(
        pl.col('articles').map_elements(
            lambda arts: ' '.join([article_id_reverse[a] for a in arts]),
            return_dtype=pl.String
        ).alias('prediction')
    )
    .with_columns(
        pl.col('customer_id').map_elements(
            lambda cid: customer_id_reverse[cid],
            return_dtype=pl.String
        ).alias('customer_id')
    )
    .select(['customer_id', 'prediction'])
)

# 全ユーザーにマージ（fallback付き）
# bin_popularのfallback
bin_pop_df = sub_cand_dfs['bin_popular']
fallback_df = (
    bin_pop_df
    .sort(['customer_id', 'rank'])
    .group_by('customer_id')
    .agg(pl.col('article_id').head(12).alias('articles'))
    .with_columns(
        pl.col('articles').map_elements(
            lambda arts: ' '.join([article_id_reverse[a] for a in arts]),
            return_dtype=pl.String
        ).alias('fallback_prediction')
    )
    .with_columns(
        pl.col('customer_id').map_elements(
            lambda cid: customer_id_reverse[cid],
            return_dtype=pl.String
        ).alias('customer_id')
    )
    .select(['customer_id', 'fallback_prediction'])
)

# 全ユーザーDF
all_customers_df = pd.DataFrame({'customer_id': list(customer_id_map.keys())})

# マージ
submission_pd = submission.to_pandas()
fallback_pd = fallback_df.to_pandas()

result = all_customers_df.merge(submission_pd, on='customer_id', how='left')
result = result.merge(fallback_pd, on='customer_id', how='left')
result['prediction'] = result['prediction'].fillna(result['fallback_prediction'])

# 最終fallback（bin_popularにもない場合）
global_fallback = (
    bin_pop_df.sort('rank').select('article_id').head(12).to_series().to_list()
)
global_fallback_str = ' '.join([article_id_reverse[a] for a in global_fallback])
result['prediction'] = result['prediction'].fillna(global_fallback_str)

result = result[['customer_id', 'prediction']]

# 保存
OUTPUT_DIR = Path('/workspace/outputs/submissions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
result.to_csv(OUTPUT_DIR / 'submission_candidate_only.csv', index=False)
print(f"保存完了: submission_candidate_only.csv")
print(f"ユーザー数: {len(result):,}")
print(result.head(5))

提出ファイル作成中...


/tmp/ipykernel_1291801/4151585914.py:20: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("customer_id").map_elements(lambda cid: ...)
with this one instead:
  + pl.col("customer_id").replace_strict(customer_id_reverse)

  pl.col('customer_id').map_elements(
/tmp/ipykernel_1291801/4151585914.py:43: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("customer_id").map_elements(lambda cid: ...)
with this one instead:
  + pl.col("customer_id").replace_strict(customer_id_reverse)

  pl.col('customer_id').map_elements(


保存完了: submission_candidate_only.csv
ユーザー数: 1,371,980
                                         customer_id  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   

                                          prediction  
0  0568601043 0779781015 0858856005 0751471001 05...  
1  0826211002 0723529001 0351484002 0800436010 09...  
2  0794321007 0794321011 0915526001 0794321008 09...  
3  0742079001 0732413001 0751471043 0751471001 08...  
4  0730683050 0791587015 0896152002 0927530004 08...  
